# Caso G · 02 Reglas Flux sobre la capa plata

> _Tutorial · Caso de uso: **G — Calidad con agentes** · Capa Medallion: **plata** · Spec: `docs/specs/synthetic-bms/02-domain-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Validar la capa plata directamente con queries Flux: completitud, rangos, presencia de los 5 tags, ausencia de variables sin metadata.


## 2. Qué se aprende

- Cómo escribir reglas Flux concisas.
- Cómo automatizar el chequeo desde Python.
- Reglas críticas vs warnings.


## 3. Contexto del caso de uso

Cuando los demás equipos cargan plata, G debe avisar de problemas. Las reglas viven en repo y se ejecutan periódicamente.


## 4. Relación con CENTINELA+

Las mismas reglas correrán contra `simarro-prod`.


## 5. Relación con Medallion

Plata.


## 6. Datos de entrada

InfluxDB (real o mock).


## 7. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [1]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


ROOT=C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS, SEED=42, default_bucket=telemetry


## 8. Schema CAPTIA esperado

Las 5 tags + field value.


## 9. Carga de datos o mock

Si Influx vivo, ejecutamos. Si no, definimos las queries para revisión.


In [2]:
flux_queries = {
    "completitud_co2": '''
from(bucket:"telemetry") |> range(start: -1d)
  |> filter(fn:(r) => r._measurement=="captia_point" and r.variable=="co2")
  |> count()
''',
    "rango_co2": '''
from(bucket:"telemetry") |> range(start: -1d)
  |> filter(fn:(r) => r.variable=="co2")
  |> filter(fn:(r) => r._value < 300 or r._value > 5000)
  |> count()
''',
    "presencia_tags": '''
schema.measurementTagKeys(bucket:"telemetry", measurement:"captia_point")
''',
    "metadata_pobladas": '''
from(bucket:"captia_metadata") |> range(start:-30d)
  |> filter(fn:(r) => r._measurement=="captia_point_meta")
  |> distinct(column:"variable")
  |> count()
''',
}
print(list(flux_queries.keys()))


['completitud_co2', 'rango_co2', 'presencia_tags', 'metadata_pobladas']


## 10. Exploración paso a paso

Ejecutamos si tenemos cliente.


In [3]:
client = get_influx_client()
results = {}
if client is not None:
    org = os.environ.get("INFLUXDB_ORG", "captia")
    for name, q in flux_queries.items():
        try:
            res = client.query_api().query_data_frame(q, org=org)
            results[name] = res
        except Exception as e:
            results[name] = f"error: {e}"
else:
    print("Modo offline: las queries quedan documentadas; `make demo` y re-ejecutar.")
results


Modo offline: las queries quedan documentadas; `make demo` y re-ejecutar.


{}

## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Reporte JSON.


## 13. Visualizaciones explicativas

Tabla — hits por regla.


## 14. Validaciones

Si tenemos cliente, ninguna regla 'rango' devuelve filas (=0 fuera de rango).


In [4]:
import os

if client is not None and isinstance(results.get("rango_co2"), pd.DataFrame):
    df = results["rango_co2"]
    print("Filas fuera rango CO2:", df["_value"].iloc[0] if len(df) else 0)


## 15. Errores comunes

1. Olvidar `range(start)` — Flux pide ventana.
2. Filtrar por `_field` cuando solo hay `value`.
3. No agrupar por aula/variable.


## 16. Ejercicios propuestos

1. Añade regla 'no_state_in_telemetry'.
2. Construye una vista que enumere variables no metadatadas.
3. Convierte la query en Flux Task con notification.


## 17. Cómo se reutiliza con datos reales

Las queries son las mismas; la dashboard de calidad es transversal.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `07_case_G_data_quality_agents/03_reglas_calidad_oro_ml.ipynb`.
- Documento web del caso: `docs/validation/data-quality.md`.


## 19. Marco teórico (nivel doctoral)

### Reglas de calidad jerárquicas

Sea $\mathcal{D}_b$ bronce, $\mathcal{D}_s$ plata, $\mathcal{D}_o$ oro.
Score por capa:

$$
\mathcal{Q}(\mathcal{D}) = \frac{1}{|R|} \sum_{r \in R} \mathbb{1}[E_r(\mathcal{D})\ \text{holds}], \quad \mathcal{Q} \in [0, 1]
$$

| Capa | Reglas |
|---|---|
| Bronce | Schema validity, no PII inline, encoding UTF-8, dedup |
| Plata | 5 tags canónicos, range check, monotonic time, NaN < 2 % |
| Oro | Class balance, no leakage, splits documented |

### Drift detection — KL divergence

$$
D_{KL}(P \parallel Q) = \sum_x P(x) \log \frac{P(x)}{Q(x)}
$$

aplicado entre histogramas $P$ (training) y $Q$ (production). Alerta si
$D_{KL} > 0.1$.

### Agentes especialistas (LLM con tools)

$$
\text{Agent}_i = \langle \pi_i, \mathcal{T}_i, \mathcal{M}_i \rangle
$$

con $\pi_i$ política (prompt), $\mathcal{T}_i$ toolkit, $\mathcal{M}_i$
memoria. Composición vía pipeline:

$$
\text{Output} = \pi_n(\pi_{n-1}(\cdots \pi_1(\text{Input})))
$$


## 20. Visión corporativa CAPTIA

### Propuesta de valor

Calidad de datos es **transversal**: sin ella ningún caso de uso tiene valor. Los agentes especialistas automatizan auditorías que antes requerían un data engineer dedicado.

### ROI estimado

| Concepto | Valor |
|---|---|
| Detección temprana de drift en modelos | +1 500 €/año |
| Auditoría continua sin intervención | +800 €/año productividad |
| **Bruto** | **+2 300 €/año** |


## 21. Bibliografía y referencias

- Schelter, S. et al. (2018). *Automating Large-Scale Data Quality Verification*. VLDB.
- Great Expectations. *Documentation*. https://greatexpectations.io
- Anthropic (2024). *Claude API — Tools*. https://docs.anthropic.com
- Polyzotis, N. et al. (2017). *Data Lifecycle Challenges in Production Machine Learning*. SIGMOD.
